# Empirical Goal Simulation for Soccer Leagues

This notebook builds per-season empirical distributions of home and away goals for each league,
then simulates match scorelines by sampling from those distributions. It preserves the original
match schema and replaces only the following columns with simulated values:

- `hometeamgoals`
- `awayteamgoals`
- `hometeamresult`
- `home_team_points`
- `away_team_points`

Outputs are written to `../csv/simulated/{league}_simulated_matches.csv`.

In [1]:
from pathlib import Path
from typing import Dict, Hashable, Mapping
import numpy as np
import pandas as pd

# Paths (run this notebook from data/european_soccer/code)
BASE_CSV = Path('../csv').resolve()
PROCESSED = BASE_CSV / 'processed'
SIMULATED = BASE_CSV / 'simulated'
SIMULATED.mkdir(parents=True, exist_ok=True)

LEAGUE_FILES = {
    'bundesliga': PROCESSED / 'bundesliga_std.csv',
    'la_liga': PROCESSED / 'la_liga_std.csv',
    'premier_league1': PROCESSED / 'premier_league1_std.csv',
    'serie_a': PROCESSED / 'serie_a_std.csv',
}

# Reproducible RNG
RNG = np.random.default_rng(42)

## Usage
- Run this notebook from `data/european_soccer/code`.
- It reads processed match tables in `../csv/processed/*.csv`.
- It writes simulated matches to `../csv/simulated/{league}_simulated_matches.csv`.

## Assumptions
- Distributions are season-specific (no pooling across seasons).
- Home and away goal counts are sampled independently from their empirical marginals.
- If a season has no occurrences for a particular goal count, the probability mass at that count is 0.
- If a season is empty or has malformed goal data, a uniform fallback over the observed range is used.

In [2]:
def empirical_goal_dists(
    df: pd.DataFrame,
    season_col: str = 'season',
    hg_col: str = 'hometeamgoals',
    ag_col: str = 'awayteamgoals'
) -> Dict[Hashable, Dict[str, np.ndarray]]:
    """Build per-season empirical goal distributions.

    Parameters
    ----------
    df : pd.DataFrame
        Input match table containing at least `season_col`, `hg_col`, `ag_col`.
    season_col : str, default 'season'
        Column indicating the season identifier.
    hg_col : str, default 'hometeamgoals'
        Column with home-team goals (numeric).
    ag_col : str, default 'awayteamgoals'
        Column with away-team goals (numeric).

    Returns
    -------
    Dict[Hashable, Dict[str, np.ndarray]]
        Mapping season -> { 'goals': array([0..K]), 'home_p': probs, 'away_p': probs }.
        Probabilities sum to 1 for each of home_p and away_p.

    Notes
    -----
    - Goal values are coerced to non-negative integers via round()->int.
    - If a season yields zero counts (edge case), uses a uniform fallback over 0..K.
    - K is the max observed goal count across home/away for that season.
    """
    dists = {}
    # Defensive copy and ensure integer goal values
    tmp = df[[season_col, hg_col, ag_col]].copy()
    # Some inputs store goals as floats; cast to int safely
    tmp[hg_col] = tmp[hg_col].astype(float).round().astype(int)
    tmp[ag_col] = tmp[ag_col].astype(float).round().astype(int)

    for season, grp in tmp.groupby(season_col):
        hg_counts = grp[hg_col].value_counts().sort_index()
        ag_counts = grp[ag_col].value_counts().sort_index()
        max_g = int(max(hg_counts.index.max(), ag_counts.index.max()))
        goals = np.arange(0, max_g + 1)
        hp = hg_counts.reindex(goals, fill_value=0).to_numpy(dtype=float)
        ap = ag_counts.reindex(goals, fill_value=0).to_numpy(dtype=float)
        hp_sum = hp.sum()
        ap_sum = ap.sum()
        if hp_sum == 0:
            hp = np.ones_like(goals, dtype=float)
            hp_sum = hp.sum()
        if ap_sum == 0:
            ap = np.ones_like(goals, dtype=float)
            ap_sum = ap.sum()
        dists[season] = {
            'goals': goals,
            'home_p': (hp / hp_sum),
            'away_p': (ap / ap_sum),
        }
    return dists


def simulate_matches_from_empirical(
    df: pd.DataFrame,
    dists: Mapping[Hashable, Dict[str, np.ndarray]],
    season_col: str = 'season'
) -> pd.DataFrame:
    """Sample goals per match using per-season empirical distributions and compute results/points.

    Parameters
    ----------
    df : pd.DataFrame
        Match table; all columns are preserved, and the following are overwritten:
        `hometeamgoals`, `awayteamgoals`, `hometeamresult`, `home_team_points`, `away_team_points`.
    dists : Mapping[Hashable, Dict[str, np.ndarray]]
        Output of `empirical_goal_dists`. Must contain an entry for each match season.
    season_col : str, default 'season'
        Season column name.

    Returns
    -------
    pd.DataFrame
        Copy of input with simulated goals/results/points.

    Raises
    ------
    KeyError
        If a match references a season not present in `dists`.
    """
    out = df.copy()

    sim_home = []
    sim_away = []
    seasons = out[season_col].tolist()
    for s in seasons:
        dist = dists[s]
        g = dist['goals']
        sim_home.append(RNG.choice(g, p=dist['home_p']))
        sim_away.append(RNG.choice(g, p=dist['away_p']))

    out['hometeamgoals'] = np.asarray(sim_home, dtype=int)
    out['awayteamgoals'] = np.asarray(sim_away, dtype=int)
    diff = out['hometeamgoals'] - out['awayteamgoals']
    out['hometeamresult'] = np.sign(diff).astype(int)
    out['home_team_points'] = np.where(diff > 0, 3, np.where(diff == 0, 1, 0)).astype(float)
    out['away_team_points'] = np.where(diff < 0, 3, np.where(diff == 0, 1, 0)).astype(float)

    return out


def simulate_and_write(in_path: Path, out_path: Path) -> Path:
    """Convenience wrapper: read processed matches, simulate, and write CSV.

    Parameters
    ----------
    in_path : Path
        Path to processed league CSV (e.g., `processed/bundesliga_std.csv`).
    out_path : Path
        Destination path for simulated matches CSV. Parent directory is created if needed.

    Returns
    -------
    Path
        The written output path.
    """
    df = pd.read_csv(in_path)
    dists = empirical_goal_dists(df, 'season', 'hometeamgoals', 'awayteamgoals')
    sim = simulate_matches_from_empirical(df, dists, 'season')
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sim.to_csv(out_path, index=False)
    return out_path

In [3]:
# Run for all leagues listed above
written = []
for league, in_fp in LEAGUE_FILES.items():
    out_fp = SIMULATED / f'{league}_simulated_matches.csv'
    path = simulate_and_write(in_fp, out_fp)
    written.append(str(path))

written

['/Users/mikeyautorino/Desktop/skillvsluck/data/european_soccer/csv/simulated/bundesliga_simulated_matches.csv',
 '/Users/mikeyautorino/Desktop/skillvsluck/data/european_soccer/csv/simulated/la_liga_simulated_matches.csv',
 '/Users/mikeyautorino/Desktop/skillvsluck/data/european_soccer/csv/simulated/premier_league1_simulated_matches.csv',
 '/Users/mikeyautorino/Desktop/skillvsluck/data/european_soccer/csv/simulated/serie_a_simulated_matches.csv']